# Code mẫu — Sentence embedding

Một hệ thống tìm tài liệu phải biến các vector thành một thứ hạng. Nó cũng phải biết lúc nào không có tín hiệu gì để xếp hạng. Các vector dưới đây được đặt bằng tay để kiểm phép toán. Chúng không phải embedding lấy từ một model pretrained.

## Ví dụ 1: Độ dài vector làm kết quả tìm kiếm lệch

Câu truy vấn có vector `(1, 1)`. Tài liệu đầu có vector `(2, 2)`, cùng hướng với truy vấn. Tài liệu thứ hai có vector `(10, 0)`, dài hơn nhiều nhưng lệch hướng. Tài liệu thứ ba ngược hướng hẳn. Ta so tích vô hướng với cosine.

In [ ]:
import numpy as np

def unit_rows(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("Không tính cosine cho vector 0")
    return vectors / norms

query = np.array([[1., 1.]])
documents = np.array([[2., 2.], [10., 0.], [-1., -1.]])
dot = query @ documents.T
cosine = unit_rows(query) @ unit_rows(documents).T
print("dot chưa chuẩn hóa:", dot.round(3).tolist())
print("cosine:", cosine.round(3).tolist())
print("top-1 dot/cosine:", int(dot.argmax()), int(cosine.argmax()))
assert np.allclose(cosine, [[1, 1 / np.sqrt(2), -1]])
assert np.allclose(unit_rows(7 * query) @ unit_rows(documents).T, cosine)
try:
    unit_rows(np.array([[0., 0.]]))
except ValueError as error:
    print("query rỗng:", error)

dot chưa chuẩn hóa: [[4.0, 10.0, -2.0]]
cosine: [[1.0, 0.707, -1.0]]
top-1 dot/cosine: 1 0
query rỗng: Không tính cosine cho vector 0


Tích vô hướng xếp tài liệu thứ hai lên đầu, chỉ vì vector của nó dài. Cosine xếp tài liệu đầu lên đầu, vì nó cùng hướng hoàn toàn với truy vấn, cosine bằng 1. Dòng assert thứ hai kiểm rằng nhân truy vấn lên 7 lần cũng không làm đổi cosine: cosine chỉ quan tâm hướng.

Vector 0 không có hướng, nên không tính được cosine. Hàm chặn ngay và báo lỗi rõ ràng, thay vì sinh ra NaN, hay lặng lẽ trả về tài liệu đầu tiên.

## Ví dụ 2: Thêm PAD không được làm đổi phép lấy trung bình

Một câu có hai token thật, với vector ẩn là `(2, 0)` và `(0, 2)`. Ta thêm hai ô PAD, và cố tình gán cho chúng giá trị `(9, 9)` để lỗi dễ thấy. Trong encoder thật, đầu ra ở vị trí PAD cũng không nhất thiết bằng 0.

In [ ]:
import torch

torch.set_num_threads(1)

def masked_mean(hidden, keep_mask):
    if hidden.shape[:2] != keep_mask.shape:
        raise ValueError("Mask phải khớp hai trục batch và token")
    if not keep_mask.any(dim=1).all():
        raise ValueError("Có câu toàn PAD")
    weights = keep_mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * weights).sum(dim=1) / weights.sum(dim=1)

real_tokens = torch.tensor([[[2., 0.], [0., 2.]]])
padded = torch.cat([real_tokens, torch.full((1, 2, 2), 9.)], dim=1)
mask = torch.tensor([[True, True, False, False]])
correct = masked_mean(padded, mask)
print("mean cả PAD:", padded.mean(dim=1).tolist())
print("mean có mask:", correct.tolist())
print("shape sau pooling:", tuple(correct.shape))
print("thêm PAD không đổi:", torch.allclose(correct, masked_mean(real_tokens, mask[:, :2])))
assert torch.allclose(correct, torch.tensor([[1., 1.]]))
try:
    masked_mean(padded, torch.zeros_like(mask))
except ValueError as error:
    print("từ chối đầu vào:", error)

mean cả PAD: [[5.0, 5.0]]
mean có mask: [[1.0, 1.0]]
shape sau pooling: (1, 2)
thêm PAD không đổi: True
từ chối đầu vào: Có câu toàn PAD


Lấy trung bình cả bốn ô cho `(5, 5)`, bị hai ô PAD kéo lệch hẳn. Có mask thì ra `(1, 1)`, vì chỉ cộng hai token thật và chia cho 2. Kết quả này giống hệt khi tính trên câu không có PAD.

Mask có shape `(1, 4)`. Ta thêm một trục cuối để nó thành `(1, 4, 1)`, nhờ vậy cùng một trọng số áp lên cả hai chiều của vector ẩn. Hàm còn kiểm hai điều trước khi tính: mask phải khớp shape với dữ liệu, và câu nào cũng phải có ít nhất một token thật.

## Ví dụ 3: Phát hiện một thứ hạng chỉ đúng nhờ may

Ba tài liệu nói về tài khoản, thư viện, và lịch học. Mỗi tài liệu là một vector chỉ về một hướng chủ đề. Ba câu truy vấn, mỗi câu có đúng một tài liệu đúng. Ta xếp hạng, rồi đo ba chỉ số: Recall@1, Recall@2, và MRR. Sau đó ta xem chuyện gì xảy ra khi một truy vấn cho điểm 0 với mọi tài liệu.

In [ ]:
import numpy as np

document_vectors = np.eye(3)
query_vectors = np.array([[0.8, 0.2, 0.0], [0.0, 0.3, 0.7], [0.1, 0.2, 0.7]])
gold = np.array([0, 1, 2])
query_vectors /= np.linalg.norm(query_vectors, axis=1, keepdims=True)
scores = query_vectors @ document_vectors.T

def retrieval_metrics(scores, gold):
    order = np.argsort(-scores, axis=1, kind="stable")
    # stable chỉ quy định cách phá hòa, không thêm bằng chứng nào về độ liên quan.
    ranks = np.array([np.flatnonzero(row == label)[0] + 1
                      for row, label in zip(order, gold)])
    return ranks, (ranks <= 1).mean(), (ranks <= 2).mean(), (1 / ranks).mean()

ranks, r1, r2, mrr = retrieval_metrics(scores, gold)
print("score:", scores.round(3).tolist())
print("hạng tài liệu đúng:", ranks.tolist())
print(f"R@1={r1:.3f} R@2={r2:.3f} MRR={mrr:.3f}")
assert ranks.tolist() == [1, 2, 1]
assert np.isclose(mrr, (1 + 1 / 2 + 1) / 3)

empty_scores = np.zeros((1, 3))
print("argmax khi toàn 0:", int(empty_scores.argmax()))
print("có tín hiệu để trả kết quả:", bool(np.any(empty_scores != 0)))

score: [[0.97, 0.243, 0.0], [0.0, 0.394, 0.919], [0.136, 0.272, 0.953]]
hạng tài liệu đúng: [1, 2, 1]
R@1=0.667 R@2=1.000 MRR=0.833
argmax khi toàn 0: 0
có tín hiệu để trả kết quả: False


Truy vấn thứ hai cần tài liệu thư viện, nhưng lại xếp tài liệu lịch học lên trên, nên tài liệu đúng đứng hạng 2. Recall@1 là 2/3, vì chỉ hai truy vấn có tài liệu đúng ở hạng đầu. Recall@2 là 1, vì cả ba đều nằm trong top 2. MRR lấy trung bình của nghịch đảo hạng: $(1+1/2+1)/3\approx0.833$.

Hai dòng cuối là cái bẫy. Khi mọi điểm đều bằng 0, `argmax` vẫn trả về 0, tức tài liệu đầu tiên, chỉ vì nó đứng đầu danh sách. Không có bằng chứng nào cho thấy tài liệu ấy liên quan. Một hệ thống tìm kiếm thật phải kiểm trường hợp không có tín hiệu này, và trả về “không tìm thấy”, thay vì bịa ra một kết quả.